In [0]:
import requests
import json
import os
from datetime import datetime

from pathlib import Path
from typing import Dict, List, Optional

import time

#loop through list mds-reference 
def update_offset(recordLimit: int, offset: int, TotalCount: int)-> str: 
        if recordLimit < TotalCount:
            offset = offset + recordLimit
            return offset
        return offset 
    
def timeout_api_restriction(responseCode: int )->None:
        if responseCode == 503 or responseCode == 504 or responseCode == 429:
            print(f"api restriction: {responseCode}")
            time.sleep(4)
        
def versioning_fileNames(filename: str) -> str:
        """ differ by milliseconds eg. airports_2026-03-12-15-34-21-482"""
        timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S-%f")[:-3]
        versioned_filename = f"{filename}_{timestamp}"
        return versioned_filename
    
def loop_until_data_pool_finished(Totaldata: int, recordLimit: int)->bool:
        if Totaldata > recordLimit:
            notfinished = True
        else:
            notfinished = False

# def setAirportsbyLufthansa( onlyLufthansa: bool) -> str:
#         if(onlyLufthansa):
#             return "?LHoperated=1"
#         else:
#             return "?LHoperated=0"

def get_data_Reference_airport(
    base_Url: str,
    headers: Dict[str, str],
    catalog_name: str,
    schema_name: str,
    volume_name: str,

    mds_reference: str,

    recordLimit: int, 
    offset: int,


    ):
    """ Get Lufthansa operated Airports only """

    
    # LHoperation = setAirportsbyLufthansa( onlyLufthansa=True)

    dic = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{mds_reference}/"
    FileName_base = f"{dic}/{mds_reference}.json"
    endpoint = f"/v1/mds-references/{mds_reference}limit={recordLimit}&offset={offset}"
    # endpoint = f"/v1/mds-references/{mds_reference}{LHoperation}limit={recordLimit}&offset={offset}"

   
    # GET /mds-references/airports[?limit={recordLimit}][&][offset={recordOffset}]

    #endpoint = f"/mds-references/airports/{airportCode}[?lang={languageCode}]",
    #endpoint = f"/msds-reference/{reference_type}?limit{limit}&offset={offset}",
    
    print(Path)
    if not os.path.exists(dic):
        os.makedirs(dic)
    
    dummy_count = 0
    notfinished = True
    
    while notfinished:
        print(f"sending request {dummy_count}")
        print(f"{base_Url}{endpoint}")

        response = requests.get(
            base_Url + endpoint,
            headers=headers)
        timeout_api_restriction(response.status_code)
        if (response.status_code == 200):
            print("Api call requests worked ")
            json_data = response.json()
            print(json_data)
            # Totaldata = json_data.get("Meta").get("TotalCount")
            # print(f"Total data: {Totaldata}")
            # offset = update_offset(recordLimit, offset, Totaldata)
            # endpoint = f"/v1/mds-references/{mds_reference}limit={recordLimit}&offset={offset}"
            
            FileName = versioning_fileNames(FileName_base)
            with open (FileName, "w") as file:
                json.dump(json_data, file, indent=2)
            print(json_data)
        else:
            print(f"new error code: {response.status_code}")
        
        # notfinished = loop_until_data_pool_finished(Totaldata, recordLimit)
        if notfinished is True:
            break   
        






In [0]:
from pathlib import Path


password = dbutils.secrets.get(scope="lh-api", key="password")
base_url = "https://lh-proxy.onrender.com"
headers ={"password": "DataIntelligence2026"}
#headers ={"password": password}
    
    
catalog_name = "data_catalog"
schema_name = "bronze"
volume_name = "bronze_volume"
    
    
    # Option 1: Fetch all reference data

#reference_type = ["mds-reference", "Offers"]
#Offers = [ "SeatMaps", "Lounges"]

#SeatMaps = ["flightNumber", "origin", "destination", "departureDate", "cabinTypeCode"]
#lounges = ["code", "cabinClassCode", "tierCode", "languageCode"]

#mds_reference = ["Countries", "Cities", "Airports", "NearestAirport", "Aircrafts"]
mds_reference ="airports"
recordLimit = 50
offset =0


print("hallo")
get_data_Reference_airport(
    base_Url="https://lh-proxy.onrender.com",
    headers=headers,
    
    catalog_name = catalog_name,
    schema_name = schema_name,
    volume_name = volume_name,

    mds_reference= mds_reference,
    
    recordLimit= recordLimit,
    offset={offset}

    )
    



In [0]:
from pathlib import Path


password = dbutils.secrets.get(scope="lh-api", key="password")


def get_all_references()->None :
    get_data_Countries(

    )
    # gett_all_Cities(
    # )
    get_data_Reference_airport()
    
